In [1]:
import numpy as np

def load_adjacency_matrix(filepath):
    # Nodes are 1-indexed -> shift to 0-indexed
    edges = np.loadtxt(filepath, delimiter=",", dtype=int) - 1
    n = edges.max() + 1
    A = np.zeros((n, n), dtype=np.float64)
    A[edges[:, 0], edges[:, 1]] = 1.0
    return A

def compute_pagerank(A, alpha=0.2, num_iter=100, tol=1e-10):
    n = A.shape[0]
    out_deg = A.sum(axis=1, keepdims=True)
    
    # M = A with rows divided by out-degree (zeros where node has no outgoing links)
    M = np.divide(A, out_deg, out=np.zeros_like(A), where=(out_deg > 0))
    
    # Page without outgoing links has alpha = 1.0, others have alpha = 0.2
    alpha_vec = np.where(out_deg > 0, alpha, 1.0)
    E = np.ones((n, n), dtype=np.float64)
    A_prime = (1.0 - alpha_vec) * M + (alpha_vec / n) * E
    A_prime_T = A_prime.T

    p = np.full(n, 1.0 / n, dtype=np.float64)
    p_100 = None
    converged_iter = None

    i = 0
    while p_100 is None or converged_iter is None:
        p_next = A_prime_T @ p
        i += 1
        if converged_iter is None and np.abs(p_next - p).sum() < tol:
            converged_iter = i
        p = p_next
        if i == num_iter:
            p_100 = p.copy()
        if i > 10000:
            break

    return p_100, converged_iter, p

def compute_hits(A, num_iter=100):
    n = A.shape[0]
    a = np.ones(n, dtype=np.float64)
    h = np.ones(n, dtype=np.float64)
    A_T = A.T

    for _ in range(num_iter):
        h_next = A @ a
        a_next = A_T @ h
        h = h_next / h_next.sum()
        a = a_next / a_next.sum()

    return a, h

results = {}
for g_id in range(1, 7):
    fname = f"graphs/graph_{g_id}.txt"
    A = load_adjacency_matrix(fname)
    pr_100, conv_iter, pr_conv = compute_pagerank(A)
    auth, hub = compute_hits(A)
    results[g_id] = {
        "A": A, "pr": pr_100, "conv_iter": conv_iter,
        "pr_conv": pr_conv, "auth": auth, "hub": hub
    }

    print(f"=== Graph {g_id} (nodes: {A.shape[0]}, edges: {int(A.sum())}) ===")
    print(f"PageRank convergence iterations (< 1e-10): {conv_iter} (100 iterations sufficient? {'YES' if conv_iter <= 100 else 'NO'})")
    if g_id <= 4:
        print("PageRank: ", np.round(pr_100, 4))
        print("Authority:", np.round(auth, 4))
        print("Hub:      ", np.round(hub, 4))
    print()

=== Graph 1 (nodes: 6, edges: 5) ===
PageRank convergence iterations (< 1e-10): 43 (100 iterations sufficient? YES)
PageRank:  [0.0656 0.1181 0.1601 0.1937 0.2205 0.242 ]
Authority: [0.  0.2 0.2 0.2 0.2 0.2]
Hub:       [0.2 0.2 0.2 0.2 0.2 0. ]

=== Graph 2 (nodes: 5, edges: 5) ===
PageRank convergence iterations (< 1e-10): 1 (100 iterations sufficient? YES)
PageRank:  [0.2 0.2 0.2 0.2 0.2]
Authority: [0.2 0.2 0.2 0.2 0.2]
Hub:       [0.2 0.2 0.2 0.2 0.2]

=== Graph 3 (nodes: 4, edges: 6) ===
PageRank convergence iterations (< 1e-10): 26 (100 iterations sufficient? YES)
PageRank:  [0.1786 0.3214 0.3214 0.1786]
Authority: [0.191 0.309 0.309 0.191]
Hub:       [0.191 0.309 0.309 0.191]

=== Graph 4 (nodes: 7, edges: 18) ===
PageRank convergence iterations (< 1e-10): 27 (100 iterations sufficient? YES)
PageRank:  [0.2726 0.1567 0.1384 0.1092 0.1853 0.0656 0.0722]
Authority: [0.1395 0.1779 0.2008 0.1402 0.2014 0.0561 0.0841]
Hub:       [0.2755 0.0478 0.1087 0.1987 0.1837 0.1167 0.069 ]

===

In [2]:
A6 = results[6]["A"]
pr6 = results[6]["pr"]
auth6 = results[6]["auth"]
hub6 = results[6]["hub"]

in_deg6 = A6.sum(axis=0).astype(int)   # incoming links (in-degree)
out_deg6 = A6.sum(axis=1).astype(int)  # outgoing links (out-degree)

def print_top5(title, scores, degree_vec, deg_label):
    top_idx = np.argsort(scores)[::-1][:5]
    print(f"--- {title} ---")
    for rank, idx in enumerate(top_idx, 1):
        # idx + 1 to restore 1-based node indexing
        print(f"{rank}. node {idx + 1:4d} | score: {scores[idx]:.6f} | {deg_label}: {degree_vec[idx]} (in={in_deg6[idx]}, out={out_deg6[idx]})")
    print()

print_top5("Top 5 PageRank (Graph 6)", pr6, in_deg6, "incoming links")
print_top5("Top 5 Authority (Graph 6)", auth6, in_deg6, "incoming links")
print_top5("Top 5 Hub (Graph 6)", hub6, out_deg6, "outgoing links")

--- Top 5 PageRank (Graph 6) ---
1. node 1052 | score: 0.003628 | incoming links: 89 (in=89, out=33)
2. node  761 | score: 0.002959 | incoming links: 68 (in=68, out=0)
3. node 1151 | score: 0.002959 | incoming links: 68 (in=68, out=0)
4. node   62 | score: 0.002948 | incoming links: 68 (in=68, out=15)
5. node   78 | score: 0.002877 | incoming links: 67 (in=67, out=28)

--- Top 5 Authority (Graph 6) ---
1. node  761 | score: 0.030401 | incoming links: 68 (in=68, out=0)
2. node 1151 | score: 0.030401 | incoming links: 68 (in=68, out=0)
3. node   62 | score: 0.030175 | incoming links: 68 (in=68, out=15)
4. node   78 | score: 0.030028 | incoming links: 67 (in=67, out=28)
5. node  394 | score: 0.029318 | incoming links: 65 (in=65, out=0)

--- Top 5 Hub (Graph 6) ---
1. node  171 | score: 0.016150 | outgoing links: 47 (in=10, out=47)
2. node  857 | score: 0.015518 | outgoing links: 30 (in=1, out=30)
3. node  185 | score: 0.015417 | outgoing links: 27 (in=1, out=27)
4. node   91 | score: 0.01

**1. Konvergence PageRanku:**
PageRank konverguje po 43, 1, 26, 27, 28 a 22 iteracích, takže 100 iterací s velkou rezervou stačí i pro největší grafy 5 a 6.

**2. Graf 6 – porovnání s odkazy a rozdíl mezi PageRankem a autoritou:**
Nejlepší uzly mají nejvíce příchozích či odchozích odkazů, přičemž uzel 1052 sice vede v příchozích odkazech i PageRanku, ale v top autoritách chybí. PageRank dělí váhu odchozími odkazy a používá teleportaci, zatímco autorita v HITS sčítá hub skóre odkazujících stránek bez dělení.

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report

def evaluate_html_lm(npz_path, label):
    data = np.load(npz_path)
    X = data["X"]
    y = data["y"]
    splits = data["splits"]

    nan_mask = np.isnan(X).any(axis=1)
    nan_count = nan_mask.sum()
    print(f"[{label}] NaN rows check: {nan_count} / {len(X)}")

    # Replace any residual NaN values (e.g., 2 rows in cleaned HTML) with 0.0
    X = np.nan_to_num(X, nan=0.0)

    train_mask = (splits == "train")
    test_mask = (splits == "test")

    X_train, y_train = X[train_mask], y[train_mask]
    X_test, y_test = X[test_mask], y[test_mask]

    clf = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=3000, C=0.1)
    )
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average="macro")

    print(f"\n=== {label} ===")
    print(f"Accuracy: {acc:.4f} | Macro-F1: {macro_f1:.4f}")
    print(classification_report(y_test, y_pred, digits=4))
    return acc, macro_f1, y_test, y_pred

acc_raw, f1_raw, y_test_raw, pred_raw = evaluate_html_lm("vectors_raw.npz", "HTML-LM (Raw HTML)")
acc_clean, f1_clean, y_test_clean, pred_clean = evaluate_html_lm("vectors_clean.npz", "HTML-LM (Cleaned HTML)")

[HTML-LM (Raw HTML)] NaN rows check: 0 / 11116

=== HTML-LM (Raw HTML) ===
Accuracy: 0.7531 | Macro-F1: 0.7102
                precision    recall  f1-score   support

          Arts     0.7739    0.7672    0.7706       116
      Business     0.8276    0.8213    0.8244       263
     Computers     0.7805    0.7356    0.7574        87
         Games     0.6923    0.4286    0.5294        21
        Health     0.7794    0.8548    0.8154        62
          Home     0.6429    0.7500    0.6923        12
Kids_and_Teens     0.6569    0.7128    0.6837        94
          News     0.7143    0.7143    0.7143         7
    Recreation     0.7667    0.7823    0.7744       147
     Reference     0.7297    0.5870    0.6506        46
       Science     0.5588    0.4750    0.5135        40
      Shopping     0.7529    0.7273    0.7399        88
       Society     0.6429    0.7397    0.6879        73
        Sports     0.7692    0.8108    0.7895        74

      accuracy                         0.7531  

In [4]:
import gzip
import json
import pandas as pd
from bs4 import BeautifulSoup, FeatureNotFound
from sklearn.feature_extraction.text import TfidfVectorizer

print("Loading curlie_cs.jsonl.gz and extracting plain text via BeautifulSoup...")
pages = [json.loads(line) for line in gzip.open("curlie_cs.jsonl.gz", "rt", encoding="utf-8")]

try:
    BeautifulSoup("<html></html>", "lxml")
    parser = "lxml"
except FeatureNotFound:
    print("Warning: 'lxml' not found in local environment, falling back to 'html.parser'...")
    parser = "html.parser"

texty_train = [BeautifulSoup(p["html"], parser).get_text(" ") for p in pages if p["split"] == "train"]
y_train_tfidf = np.array([p["category"] for p in pages if p["split"] == "train"])

texty_test = [BeautifulSoup(p["html"], parser).get_text(" ") for p in pages if p["split"] == "test"]
y_test_tfidf = np.array([p["category"] for p in pages if p["split"] == "test"])

tfidf = TfidfVectorizer(min_df=2, sublinear_tf=True).fit(texty_train)
X_train_tfidf = tfidf.transform(texty_train)
X_test_tfidf = tfidf.transform(texty_test)

clf_tfidf = LogisticRegression(max_iter=3000, C=10).fit(X_train_tfidf, y_train_tfidf)
y_pred_tfidf = clf_tfidf.predict(X_test_tfidf)

acc_tfidf = accuracy_score(y_test_tfidf, y_pred_tfidf)
f1_tfidf = f1_score(y_test_tfidf, y_pred_tfidf, average="macro")

print("\n=== TF-IDF + LogisticRegression (C=10) ===")
print(f"Vocabulary size: {len(tfidf.vocabulary_)} terms")
print(f"Accuracy: {acc_tfidf:.4f} | Macro-F1: {f1_tfidf:.4f}")
print(classification_report(y_test_tfidf, y_pred_tfidf, digits=4))

summary_df = pd.DataFrame([
    {"Method": "HTML-LM (Raw HTML)",     "Accuracy": round(acc_raw, 4),   "Macro-F1": round(f1_raw, 4)},
    {"Method": "HTML-LM (Cleaned HTML)", "Accuracy": round(acc_clean, 4), "Macro-F1": round(f1_clean, 4)},
    {"Method": "TF-IDF (Plain Text)",    "Accuracy": round(acc_tfidf, 4), "Macro-F1": round(f1_tfidf, 4)},
])
display(summary_df)

Loading curlie_cs.jsonl.gz and extracting plain text via BeautifulSoup...

=== TF-IDF + LogisticRegression (C=10) ===
Vocabulary size: 168083 terms
Accuracy: 0.7646 | Macro-F1: 0.7187
                precision    recall  f1-score   support

          Arts     0.7627    0.7759    0.7692       116
      Business     0.7224    0.8707    0.7897       263
     Computers     0.7692    0.6897    0.7273        87
         Games     0.8000    0.3810    0.5161        21
        Health     0.9259    0.8065    0.8621        62
          Home     0.7500    0.7500    0.7500        12
Kids_and_Teens     0.7582    0.7340    0.7459        94
          News     1.0000    0.4286    0.6000         7
    Recreation     0.7561    0.8435    0.7974       147
     Reference     0.8286    0.6304    0.7160        46
       Science     0.7368    0.3500    0.4746        40
      Shopping     0.8442    0.7386    0.7879        88
       Society     0.7013    0.7397    0.7200        73
        Sports     0.8000    0.

,Method,Accuracy,Macro-F1
0,HTML-LM (Raw HTML),0.7531,0.7102
1,HTML-LM (Cleaned HTML),0.7071,0.6582
2,TF-IDF (Plain Text),0.7646,0.7187


**1. Nejlepší a nejhorší kategorie:**
Nejlépe vycházejí velké a specifické kategorie (*Business*, *Health*, *Sports*, *Recreation* s F1 0,77–0,86) díky dostatku dat a jasným klíčovým slovům. Nejhůře vycházejí malé nebo obecné kategorie (*Science*, *Games*, *News*, *Home* s F1 0,47–0,69), protože mají málo vzorků a jejich slovník se překrývá s ostatními tématy.

**2. Srovnání TF-IDF a HTML-LM:**
TF-IDF není horší než HTML-LM, naopak dosahuje mírně vyšší přesnosti (0,7646 vs. 0,7531) i Macro-F1 (0,7187 vs. 0,7102). Kategorie webu totiž silně závisí na konkrétních klíčových slovech z celého textu (168 tisíc slov v TF-IDF), zatímco zmrazený HTML-LM vše komprimuje do 768 dimenzí a ořezává stránky na 4096 tokenů.

**3. Vliv pročištění HTML:**
Do 4096 tokenů se před vyčištěním vešlo 10 155 / 11 116 stránek (91,35 %) a po vyčištění 10 522 / 11 116 stránek (94,66 %). Pročištění ale přesnost zhoršilo (pokles z 0,7531 na 0,7071), protože odstranění `<header>`, `<nav>`, `<footer>` a `<head>` smazalo navigační menu a titulky, které u úvodních stránek nesou nejvýstižnější popis webu.